In [39]:
from reasoning_from_scratch.qwen3 import download_qwen3_small,Qwen3Tokenizer
from reasoning_from_scratch.qwen3 import Qwen3Model,QWEN_CONFIG_06_B,KVCache
from pathlib import Path
import torch.nn as nn
import torch

In [4]:
download_qwen3_small(kind="base",tokenizer_only=False,out_dir="../qwen")

✓ ..\qwen\qwen3-0.6B-base.pth already up-to-date


In [5]:
tokenizer_path=Path("..")/"qwen"/"tokenizer-base.json"
tokenizer=Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [6]:
device=torch.device("cpu")

In [7]:
model_path = Path("..") / "qwen" / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))

<All keys matched successfully>

In [8]:
input_ids=torch.tensor(tokenizer.encode("Hey who are you"))
input_ids=input_ids.unsqueeze(0).to(device)

In [77]:

def text_gen(prompt: str, model, n_tokens, eos_token_id=151643):
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
    outputids = torch.tensor([], dtype=torch.int)
    model.eval()

    for _ in range(n_tokens):
        output = model(input_ids)
        output = output.squeeze(0)
        last_logit = output[-1, :]
        tokenid = torch.argmax(last_logit, dim=-1, keepdim=True)

        if tokenid.item() == eos_token_id:
            break

        print(tokenizer.decode(tokenid.tolist()), end="", flush=True)
        yield tokenid

        # outputids=torch.concat((outputids,tokenid),dim=-1)
        input_ids = torch.concat(
            (input_ids, tokenid.unsqueeze(0)), dim=-1
        )

    # return tokenizer.decode(list(outputids))


In [82]:

def text_gen(prompt: str,model,n_tokens,eos_token_id=151643,temperature=1.0,top_k=50):
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
    outputids = torch.tensor([], dtype=torch.int)
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    with torch.no_grad():
        output = model(input_ids, cache=cache)

        for _ in range(n_tokens):

            last_logit = output[:, -1, :].squeeze(0)

            #temp scaling
            last_logit = last_logit / temperature

            # topk sampling
            k = min(top_k, last_logit.numel())
            topk_logits, topk_indices = torch.topk(last_logit, k)
            probs = torch.softmax(topk_logits, dim=-1)
            sampled_index = torch.multinomial(probs, num_samples=1)
            tokenid = topk_indices[sampled_index].reshape(1)

            if tokenid.item() == eos_token_id:
                break

            print(tokenizer.decode(tokenid.tolist()), end="", flush=True)
            yield tokenid

            # outputids=torch.concat((outputids,tokenid),dim=-1)
            input_ids = torch.concat(
                (input_ids, tokenid.unsqueeze(0)), dim=-1
            )
            output = model(tokenid.unsqueeze(0), cache=cache)

    # return tokenizer.decode(list(outputids))


In [83]:
## with KV cache
import time

st = time.perf_counter()
n=0

for token in text_gen("What is Nepal", model, n_tokens=100):
    n=n+1

end = time.perf_counter()

print("Total time:", end - st, "seconds","with ",n,"tokens")
print("Average time per token:", (end - st) / n, "seconds")
print("Tokens per second:", n / (end - st))

's stance on ending apartheid in South Africa, and how does it plan to improve relations with South Africa? Additionally, what are Nepal's plans to counter terrorist activities in the region, and how does it support its people living in refugee camps in South Sudan and Sri Lanka?

Nepal is committed to ending apartheid in South Africa, recognizing it as a violation of international human rights law. As a member state of the United Nations and the Commonwealth, Nepal participates in multilateral actions and initiatives aimed at promotingTotal time: 14.723048099999687 seconds with  100 tokens
Average time per token: 0.14723048099999686 seconds
Tokens per second: 6.792071812901441


In [87]:
##  with kv cache

def text_gen_greedy(prompt: str, model, n_tokens, eos_token_id=151643):
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)

    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    with torch.no_grad():
        output = model(input_ids, cache=cache)

        for _ in range(n_tokens):
            last_logit = output[:, -1, :]
            tokenid = torch.argmax(last_logit, dim=-1, keepdim=True)

            if tokenid.item() == eos_token_id:
                break

            yield tokenid

            output = model(tokenid, cache=cache)


st = time.perf_counter()
n_generated = 0

for token in text_gen_greedy("What is Nepal", model, n_tokens=300):
    n_generated += 1

end = time.perf_counter()
elapsed = end - st

print("\nTotal time:", elapsed, "seconds")
print("Generated tokens:", n_generated)

if n_generated > 0:
    print("Average time per token:", elapsed / n_generated, "seconds")
    print("Tokens per second:", n_generated / elapsed)


Total time: 47.50110379999933 seconds
Generated tokens: 300
Average time per token: 0.15833701266666442 seconds
Tokens per second: 6.315642711443776


In [ ]:
### NO kv cache and high no of tokens
def text_gen(prompt: str, model, n_tokens, eos_token_id=151643):
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
    outputids = torch.tensor([], dtype=torch.int)
    model.eval()

    for _ in range(n_tokens):
        output = model(input_ids)
        output = output.squeeze(0)
        last_logit = output[-1, :]
        tokenid = torch.argmax(last_logit, dim=-1, keepdim=True)

        if tokenid.item() == eos_token_id:
            break

        print(tokenizer.decode(tokenid.tolist()), end="", flush=True)
        yield tokenid

        # outputids=torch.concat((outputids,tokenid),dim=-1)
        input_ids = torch.concat(
            (input_ids, tokenid.unsqueeze(0)), dim=-1
        )

    # return tokenizer.decode(list(outputids))
st = time.perf_counter()
n_generated = 0

for token in text_gen("What is Nepal", model, n_tokens=300):
    n_generated += 1

end = time.perf_counter()
elapsed = end - st

print("\nTotal time:", elapsed, "seconds")
print("Generated tokens:", n_generated)

if n_generated > 0:
    print("Average time per token:", elapsed / n_generated, "seconds")
    print("Tokens per second:", n_generated / elapsed)

's current status in terms of its population, and how does it compare to other countries? Additionally, what is the significance of the 2015 census in Nepal, and how does Nepal's population growth compare to other countries? Finally, what is the current population density in Nepal?

Nepal's population is currently 28.5 million, making it the 13th most populous country in the world. This places Nepal among the top 10 countries with the highest population density, which is approximately 1,000 people per square kilometer. Nepal's population growth has been relatively stable, with a growth rate of 1.1% per year, which is lower than some other countries like India, where the growth rate is around 2.5%. The 2015 census is significant because it provided the most recent and comprehensive data on Nepal's population, including its population density, which is an important indicator of urbanization and economic development. As for the current population density, it is approximately 1,000 people 